## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [196]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [197]:
# Always remember to do this!
load_dotenv(override=True)
from openai import OpenAI

In [198]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = ollama.chat.completions.create(
    model="llama3.2:latest",
    messages=messages
)

question = response.choices[0].message.content
display(Markdown(question))

## Calling LLMs from multple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [202]:
# OpenAI Compatible URLs


OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [203]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [204]:

messages = [{"role": "user", "content": question}]

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">Many models on Ollama are FAR too large for your home computer. Be sure to browse the models on the Ollama website. Look to use models that are size 3GB or less unless you know better; llama3.2 is a great first choice. Don't pick models that end in :cloud; that's something different (a cloud inference service, like Groq).
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json

result = {"question" : question ,
    "models": {}
}

for model_name in ["llama3.2", "llama3.2:latest"]:

    response = ollama.chat.completions.create(
        model=model_name,
        messages=messages
    )

    answer = response.choices[0].message.content

    result["models"][model_name] = {
        "answer": answer
    }

print(json.dumps(result, indent=2))

In [206]:
result["models"].items()

dict_items([('llama3.2', {'answer': "The distinction between experiencing a moment of profound beauty and genuinely experiencing love can be complex, but some key differences can be identified. Here are some possible distinctions:\n\n1. **Intensity vs. Depth**: A moment of profound beauty might evoke strong emotions, such as awe, wonder, or even tears, due to the intensity of the sensory experience (e.g., walking in a breathtaking sunset, gazing at a masterpiece). This type of beautiful experience is often characterized by its surface-level quality, where we respond mainly with emotional reactions. On the other hand, genuine love involves deeper, more profound connections that resonate beyond surface-level experiences.\n\n2. **Immediacy vs. Intimacy**: A moment of beauty might create an immediate sense of connection or wonder, but it may not necessarily be accompanied by a long-term, deepened bond between people. Genuine love, however, often involves establishing and nurturing deep con

In [207]:
together = ""
for index, (model_name, model_data) in enumerate(result["models"].items()):
    answer = model_data["answer"]
    together += f"# Response from model - {model_name} \n\n"
    together += answer + "\n\n"


In [ ]:
print(together)

In [ ]:
total_answers = sum(
    1 for model in result["models"].values()
    if "answer" in model
)

print(total_answers)

In [210]:
judge = f"""You are judging a competition between {total_answers} answers.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
The number of model names in "results" MUST equal the number of models provided.
Return with a valid JSON in  this format:

{{"results":["best_model_name","second_best_model_name" , "third_best_model_name"]}}

Please take these into consideration :
STRICT RULES:
- There are exactly {len(result["models"])} candidates.
- You MUST rank every candidate.
- Even if a response is completely irrelevant, it MUST be ranked.
- Never exclude a candidate.
- Never say that there is no best candidate.
- Do not discuss whether candidates come from the same model.
- Candidate numbers are identifiers only.
- Return each candidate number exactly once.
- Return ONLY valid JSON.
- Do not provide explanations.
- Do not use markdown.



Here are the responses from each models:

{together}

Now in JSON for format mention best models name  in sequence for which you have bests answers.
"""


In [ ]:
print(judge)

In [212]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
result["models"].keys()

## And now for Grok! - But done with llma3.2:latest as sample

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

In [ ]:
# Judgement time!
# Grok is "The most truth-seeking large language model in the world."

#model_name = "grok-4.3"

model_name = "llama3.2:latest"
response = ollama.chat.completions.create(model=model_name, messages=judge_messages )
responses = response.choices[0].message.content
print(responses)


In [223]:

results_dict = json.loads(results)

print(results_dict)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    print(f"Rank {index+1}: {ranks[index]}")


{'results': ['llama3.2:latest', 'llama3.2'], 'best_model': 'llama3.2:latest'}
Rank 1: llama3.2:latest
Rank 2: llama3.2


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>